# 08 Incremental Ingestion

Notebook to validate incremental ingestion and document versioning with three scenarios:

1. First run ingests the source
2. Second run skips unchanged content
3. Third run creates a new version after content changes


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

for candidate in [PROJECT_ROOT, PROJECT_ROOT.parent]:
    if (candidate / "pyproject.toml").exists():
        PROJECT_ROOT = candidate
        break

SRC_DIR = PROJECT_ROOT / "src"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed dir: {PROCESSED_DIR}")

Project root: /Users/rodrigue.lawson/VSCode Projects/lexcare-ai
Processed dir: /Users/rodrigue.lawson/VSCode Projects/lexcare-ai/data/processed


In [2]:
from __future__ import annotations

from datetime import UTC, datetime
from pathlib import Path
from unittest.mock import Mock

from app.domain.source import SourceDefinition
from app.domain.source_artifact import SourceArtifact
from app.repositories.document_repository import FileDocumentRepository
from app.repositories.document_version_repository import FileDocumentVersionRepository
from app.repositories.source_registry import SourceRegistry
from app.services.document_versioning_service import DocumentVersioningService
from app.services.incremental_ingestion_service import IncrementalIngestionService


## Notebook setup

The notebook uses a local fake source registry and a monkeypatched connector so it can be run safely without external dependencies.

In [3]:
class FakeConnector:
    def __init__(self, artifacts):
        self._artifacts = artifacts

    def fetch(self):
        return self._artifacts


class FakeDocumentRepository:
    def __init__(self) -> None:
        self.saved = []

    def save(self, document):
        stored = Mock(
            document_id=f"doc-{len(self.saved) + 1}",
            source_path=document.source_path,
            text=document.text,
            metadata=document.metadata,
        )
        self.saved.append(stored)
        return stored


source = SourceDefinition(
    source_id="test-source",
    name="Test Source",
    source_type="file",
    description="Notebook test source",
    metadata={},
)

registry = Mock(spec=SourceRegistry)
registry.get.return_value = source

version_repo_path = PROCESSED_DIR / "notebook_versions.json"
version_repo_path.parent.mkdir(parents=True, exist_ok=True)

version_repo = FileDocumentVersionRepository(storage_path=str(version_repo_path))
version_service = DocumentVersioningService(version_repository=version_repo)
doc_repo = FakeDocumentRepository()

print("Notebook services initialized.")

Notebook services initialized.


## Scenario 1 — first ingestion

The first run should ingest the document and create version 1.

In [4]:
artifact_v1 = SourceArtifact(
    source_id="test-source",
    source_type="file",
    title="doc",
    uri="file:///doc.txt",
    content="version-1",
    retrieved_at=datetime(2024, 1, 1, tzinfo=UTC),
    metadata={
        "document_type": "law",
        "topic": "test",
    },
)

import app.services.incremental_ingestion_service as incremental_module
incremental_module.ConnectorFactory.create = lambda _source: FakeConnector([artifact_v1])

service = IncrementalIngestionService(
    source_registry=registry,
    document_repository=doc_repo,
    document_versioning_service=version_service,
)

summary_1 = service.ingest_source("test-source")
summary_1

IngestionSummary(source_id='test-source', fetched=1, ingested=1, skipped=0, updated=0, errors=0)

In [5]:
versions_after_run_1 = version_service.list_versions("test-source", "file:///doc.txt")
[
    {
        "version_number": v.version_number,
        "content_hash": v.content_hash,
        "is_current": v.is_current,
        "effective_from": v.effective_from.isoformat(),
        "effective_to": v.effective_to.isoformat() if v.effective_to else None,
    }
    for v in versions_after_run_1
]

[{'version_number': 1,
  'content_hash': '12ddae32c9fd0c6969e03269ae104247c6a2a7efb4d4b37586aac8a6c76ec625',
  'is_current': True,
  'effective_from': '2024-01-01T00:00:00+00:00',
  'effective_to': None}]

## Scenario 2 — unchanged content

The second run uses the same payload, so the document should be skipped.

In [6]:
summary_2 = service.ingest_source("test-source")
summary_2

IngestionSummary(source_id='test-source', fetched=1, ingested=0, skipped=1, updated=0, errors=0)

In [7]:
versions_after_run_2 = version_service.list_versions("test-source", "file:///doc.txt")
[
    {
        "version_number": v.version_number,
        "content_hash": v.content_hash,
        "is_current": v.is_current,
        "effective_from": v.effective_from.isoformat(),
        "effective_to": v.effective_to.isoformat() if v.effective_to else None,
    }
    for v in versions_after_run_2
]

[{'version_number': 1,
  'content_hash': '12ddae32c9fd0c6969e03269ae104247c6a2a7efb4d4b37586aac8a6c76ec625',
  'is_current': True,
  'effective_from': '2024-01-01T00:00:00+00:00',
  'effective_to': None}]

## Scenario 3 — changed content

The third run changes the content, so the service should ingest a new version and close the previous one.

In [8]:
artifact_v2 = SourceArtifact(
    source_id="test-source",
    source_type="file",
    title="doc",
    uri="file:///doc.txt",
    content="version-2 with updated content",
    retrieved_at=datetime(2024, 1, 2, tzinfo=UTC),
    metadata={
        "document_type": "law",
        "topic": "test",
    },
)

incremental_module.ConnectorFactory.create = lambda _source: FakeConnector([artifact_v2])

summary_3 = service.ingest_source("test-source")
summary_3

IngestionSummary(source_id='test-source', fetched=1, ingested=0, skipped=0, updated=1, errors=0)

In [9]:
versions_after_run_3 = version_service.list_versions("test-source", "file:///doc.txt")
[
    {
        "version_number": v.version_number,
        "content_hash": v.content_hash,
        "is_current": v.is_current,
        "effective_from": v.effective_from.isoformat(),
        "effective_to": v.effective_to.isoformat() if v.effective_to else None,
    }
    for v in versions_after_run_3
]

[{'version_number': 1,
  'content_hash': '12ddae32c9fd0c6969e03269ae104247c6a2a7efb4d4b37586aac8a6c76ec625',
  'is_current': False,
  'effective_from': '2024-01-01T00:00:00+00:00',
  'effective_to': '2024-01-02T00:00:00+00:00'},
 {'version_number': 2,
  'content_hash': '3f3d4fc618e015bf0e12e14a9bf3a0179499c962a128dd37bbe1e6b7d37fc601',
  'is_current': True,
  'effective_from': '2024-01-02T00:00:00+00:00',
  'effective_to': None}]

## Summary

- Run 1: ingested
- Run 2: skipped
- Run 3: updated

The notebook can be re-run from top to bottom and will write its version history to `data/processed/notebook_versions.json`.